# 🏏 Real-Time Cricket Shot Detection
**EE655 Group Project — Live Inference Notebook**

Uses the **exact same** pipeline as the training notebook:
- `EfficientNetB0` backbone + `LSTM` / `GRU` / `CNNOnly` head
- `uniform` / `motion` / `hybrid` frame sampling
- 16-frame sliding-window buffer @ 224×224
- Sliding majority-vote smoothing + confidence threshold

**10 shot classes:** cover, defense, flick, hook, late_cut, lofted, pull, square_cut, straight, sweep

In [ ]:
# ── Cell 1: Install / verify dependencies ─────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# torchvision is already present on Kaggle; opencv usually is too.
# Uncomment if running locally:
# pip_install('torch torchvision')
# pip_install('opencv-python-headless')

import os, cv2, time, random, collections
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from IPython.display import display, Image as IPImage, clear_output
import ipywidgets as widgets
from PIL import Image
import io

print('✓ imports OK')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── Cell 2: Config (matches training notebook exactly) ─────────────────

CLASSES = ['cover', 'defense', 'flick', 'hook', 'late_cut',
           'lofted', 'pull', 'square_cut', 'straight', 'sweep']
NUM_CLASSES  = 10
FRAME_SIZE   = (224, 224)          # same as training
NUM_FRAMES   = 16                  # same as training
HIDDEN_SIZE  = 256                 # same as training
DROPOUT      = 0.65                # same as training
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Runtime controls ──────────────────────────────────────────────────
PREDICT_EVERY_N_FRAMES = 8         # run inference every N frames (lower = more responsive, higher = faster)
BUFFER_SIZE            = 32        # sliding window: keep last N raw frames before sampling
CONFIDENCE_THRESHOLD   = 0.40      # ignore predictions below 40% — show '?' instead
SMOOTH_WINDOW          = 5         # majority vote over last N predictions
SAMPLING_STRATEGY      = 'uniform' # 'uniform' | 'motion' | 'hybrid'  (uniform is fastest)

# ── Checkpoint path ───────────────────────────────────────────────────
# Set this to your saved .pth file path (Kaggle: /kaggle/input/…)
CHECKPOINT_PATH = '/kaggle/input/your-dataset/checkpoints/phase1_lstm_uniform_checkpoint.pth'
# Architecture is inferred from filename; override here if needed:
ARCH_OVERRIDE   = None  # e.g. 'lstm' | 'gru' | 'cnn_only' | None (auto)

# ── Video source ──────────────────────────────────────────────────────
# For Kaggle: use a video file.  For webcam: use 0
VIDEO_SOURCE = '/kaggle/input/your-dataset/sample_video.mp4'  # or 0 for webcam

# ── Timestamp log ─────────────────────────────────────────────────────
SAVE_TIMESTAMPS = True             # log detected shots to CSV
TIMESTAMP_CSV   = '/kaggle/working/shot_timestamps.csv'

print(f'Device  : {DEVICE}')
print(f'Strategy: {SAMPLING_STRATEGY}  |  Buffer: {BUFFER_SIZE}  |  Predict every: {PREDICT_EVERY_N_FRAMES} frames')
print(f'Classes : {CLASSES}')

In [ ]:
# ── Cell 3: Frame sampling functions (identical to training notebook) ──

def compute_motion_scores(frames):
    scores = [0.0]
    for i in range(1, len(frames)):
        g1 = cv2.cvtColor(frames[i-1], cv2.COLOR_RGB2GRAY).astype(np.float32)
        g2 = cv2.cvtColor(frames[i],   cv2.COLOR_RGB2GRAY).astype(np.float32)
        scores.append(float(np.mean(np.abs(g2 - g1))))
    return np.array(scores)


def uniform_sampling(frames, n):
    if not frames:
        return [np.zeros((*FRAME_SIZE, 3), dtype=np.uint8)] * n
    if len(frames) <= n:
        return frames + [frames[-1]] * (n - len(frames))
    return [frames[i] for i in np.linspace(0, len(frames)-1, n, dtype=int)]


def motion_sampling(frames, n):
    if len(frames) <= n:
        return frames + [frames[-1]] * (n - len(frames))
    scores = compute_motion_scores(frames)
    top_indices = set(np.argsort(scores)[-n:])
    context = set()
    for idx in top_indices:
        for off in (-3,-2,-1,1,2,3):
            nb = idx + off
            if 0 <= nb < len(frames): context.add(nb)
    all_idx = top_indices | context
    if len(all_idx) < n:
        all_idx |= set(np.linspace(0, len(frames)-1, n-len(all_idx), dtype=int))
    return [frames[i] for i in sorted(all_idx)[:n]]


def hybrid_sampling(frames, n):
    if len(frames) <= n:
        return frames + [frames[-1]] * (n - len(frames))
    m_n = int(n * 0.4); u_n = int(n * 0.4); r_n = n - m_n - u_n
    scores = compute_motion_scores(frames)
    m_idx = set(np.argsort(scores)[-m_n:])
    ctx = set()
    for idx in m_idx:
        for off in (-2,-1,1,2):
            nb = idx + off
            if 0 <= nb < len(frames): ctx.add(nb)
    m_idx |= ctx
    u_idx = set(np.linspace(0, len(frames)-1, u_n, dtype=int))
    rem = [i for i in range(len(frames)) if i not in m_idx and i not in u_idx]
    r_idx = set(np.random.choice(rem, min(r_n, len(rem)), replace=False)) if rem and r_n > 0 else set()
    all_idx = sorted(m_idx | u_idx | r_idx)
    while len(all_idx) < n: all_idx.append(all_idx[-1])
    return [frames[i] for i in all_idx[:n]]


def sample_frames(frames, n, strategy):
    if strategy == 'uniform': return uniform_sampling(frames, n)
    if strategy == 'motion':  return motion_sampling(frames, n)
    if strategy == 'hybrid':  return hybrid_sampling(frames, n)
    raise ValueError(strategy)

print('✓ Sampling functions ready')

In [ ]:
# ── Cell 4: Model definitions (identical architecture to training) ──────

class EfficientEncoder(nn.Module):
    def __init__(self, fine_tune_blocks=3, freeze_bn=False):
        super().__init__()
        base = models.efficientnet_b0(weights=None)  # weights loaded from checkpoint
        for p in base.parameters(): p.requires_grad = False
        for block in list(base.features.children())[-fine_tune_blocks:]:
            for p in block.parameters(): p.requires_grad = True
        if freeze_bn:
            for m in base.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
                    for p in m.parameters(): p.requires_grad = False
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 1280

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)


class CricketLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn  = EfficientEncoder(fine_tune_blocks=2, freeze_bn=True)
        self.lstm = nn.LSTM(input_size=1280, hidden_size=HIDDEN_SIZE,
                            num_layers=1, batch_first=True,
                            dropout=0.0, bidirectional=False)
        self.head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, 512), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(512, NUM_CLASSES))

    def forward(self, x):
        B, T, C, H, W = x.shape
        f = self.cnn(x.view(B*T, C, H, W)).view(B, T, -1)
        out, _ = self.lstm(f)
        return self.head(out[:, -1])


class CricketGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = EfficientEncoder(fine_tune_blocks=2, freeze_bn=True)
        self.gru = nn.GRU(input_size=1280, hidden_size=HIDDEN_SIZE,
                          num_layers=2, batch_first=True,
                          dropout=0.5, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE*2, 512), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(512, NUM_CLASSES))

    def forward(self, x):
        B, T, C, H, W = x.shape
        f = self.cnn(x.view(B*T, C, H, W)).view(B, T, -1)
        out, _ = self.gru(f)
        return self.head(out.mean(dim=1))


class CNNOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn  = EfficientEncoder(fine_tune_blocks=3)
        self.head = nn.Sequential(
            nn.Linear(1280, 512), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(512, NUM_CLASSES))

    def forward(self, x):
        B, T, C, H, W = x.shape
        return self.head(self.cnn(x.view(B*T, C, H, W)).view(B, T, -1).mean(dim=1))


def get_model(arch):
    return {'lstm': CricketLSTM, 'gru': CricketGRU, 'cnn_only': CNNOnly}[arch]()

print('✓ Model classes ready')

In [ ]:
# ── Cell 5: load_model() ───────────────────────────────────────────────

def infer_arch_from_name(path):
    name = os.path.basename(path).lower()
    if 'gru'  in name: return 'gru'
    if 'lstm' in name: return 'lstm'
    return 'cnn_only'


def load_model(checkpoint_path=CHECKPOINT_PATH, arch_override=ARCH_OVERRIDE):
    arch = arch_override if arch_override else infer_arch_from_name(checkpoint_path)
    print(f'Architecture : {arch}')
    print(f'Checkpoint   : {checkpoint_path}')

    model = get_model(arch).to(DEVICE)
    state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    print(f'✓ Model loaded on {DEVICE}')
    return model, arch


# ImageNet normalisation — same as training pipeline
TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(FRAME_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

MODEL, ARCH = load_model()

In [ ]:
# ── Cell 6: preprocess_frames() ────────────────────────────────────────

def preprocess_frames(raw_frames, strategy=SAMPLING_STRATEGY):
    """
    raw_frames : list of BGR numpy arrays straight from cv2.VideoCapture
    Returns    : (1, T, C, H, W) float32 tensor on DEVICE
    """
    rgb_frames = [
        cv2.cvtColor(cv2.resize(f, FRAME_SIZE), cv2.COLOR_BGR2RGB)
        for f in raw_frames
    ]
    sampled = sample_frames(rgb_frames, NUM_FRAMES, strategy)
    tensor  = torch.stack([TRANSFORM(f) for f in sampled])   # (T, C, H, W)
    return tensor.unsqueeze(0).to(DEVICE)                     # (1, T, C, H, W)


print('✓ preprocess_frames ready')

In [ ]:
# ── Cell 7: predict_shot() ─────────────────────────────────────────────

@torch.inference_mode()
def predict_shot(frame_buffer, model=None):
    """
    frame_buffer : list of raw BGR frames (sliding window)
    Returns      : (label_str, confidence_float, all_probs_np)
    """
    if model is None: model = MODEL
    tensor   = preprocess_frames(list(frame_buffer))
    logits   = model(tensor)                          # (1, NUM_CLASSES)
    probs    = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    idx      = int(np.argmax(probs))
    return CLASSES[idx], float(probs[idx]), probs


print('✓ predict_shot ready')

In [ ]:
# ── Cell 8: draw_overlay() — HUD for the live display ──────────────────

LABEL_COLOUR      = (0,   255,  80)   # bright green
CONF_HIGH_COLOUR  = (0,   255, 120)
CONF_LOW_COLOUR   = (0,   180, 255)   # amber
FPS_COLOUR        = (200, 200, 200)
BAR_BG_COLOUR     = ( 30,  30,  30)


def draw_overlay(frame, label, confidence, fps, all_probs, smoothed_label=None):
    h, w = frame.shape[:2]
    overlay = frame.copy()

    # semi-transparent header band
    cv2.rectangle(overlay, (0, 0), (w, 60), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, overlay)

    # ── primary label ──────────────────────────────────────────────────
    display_label = smoothed_label if smoothed_label else label
    if confidence < CONFIDENCE_THRESHOLD:
        display_label = '?  (low confidence)'
        colour = CONF_LOW_COLOUR
    else:
        colour = LABEL_COLOUR

    cv2.putText(overlay, f'Shot: {display_label.upper()}', (10, 30),
                cv2.FONT_HERSHEY_DUPLEX, 0.85, colour, 2, cv2.LINE_AA)

    # confidence %
    conf_text  = f'{confidence*100:.1f}%'
    conf_colour = CONF_HIGH_COLOUR if confidence >= CONFIDENCE_THRESHOLD else CONF_LOW_COLOUR
    cv2.putText(overlay, conf_text, (w - 100, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, conf_colour, 2, cv2.LINE_AA)

    # FPS
    cv2.putText(overlay, f'FPS: {fps:.1f}', (10, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, FPS_COLOUR, 1, cv2.LINE_AA)

    # ── mini bar chart — top-5 classes ─────────────────────────────────
    top5_idx = np.argsort(all_probs)[::-1][:5]
    bar_x, bar_y, bar_w, bar_h = 10, h - 110, 160, 14
    for rank, ci in enumerate(top5_idx):
        by = bar_y + rank * (bar_h + 3)
        filled = int(bar_w * all_probs[ci])
        cv2.rectangle(overlay, (bar_x, by), (bar_x + bar_w, by + bar_h), BAR_BG_COLOUR, -1)
        bar_col = (0, 255, 100) if ci == np.argmax(all_probs) else (80, 160, 255)
        cv2.rectangle(overlay, (bar_x, by), (bar_x + filled, by + bar_h), bar_col, -1)
        cv2.putText(overlay,
                    f'{CLASSES[ci][:10]:10s} {all_probs[ci]*100:4.1f}%',
                    (bar_x + bar_w + 5, by + bar_h - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (220, 220, 220), 1, cv2.LINE_AA)

    return overlay


print('✓ draw_overlay ready')

In [ ]:
# ── Cell 9: MAIN real-time inference loop ─────────────────────────────
# Works in TWO modes:
#   • Kaggle / headless → renders frames inline in the notebook
#   • Local with display → uses cv2.imshow

def run_realtime(
    source          = VIDEO_SOURCE,
    predict_every   = PREDICT_EVERY_N_FRAMES,
    buffer_size     = BUFFER_SIZE,
    conf_threshold  = CONFIDENCE_THRESHOLD,
    smooth_window   = SMOOTH_WINDOW,
    strategy        = SAMPLING_STRATEGY,
    save_timestamps = SAVE_TIMESTAMPS,
    csv_path        = TIMESTAMP_CSV,
    max_frames      = None,        # None = no limit
    display_inline  = True,        # True for Kaggle; False for local cv2.imshow
    inline_every    = 15,          # refresh inline display every N frames
):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f'ERROR: Could not open source: {source}')
        return

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    print(f'Source FPS: {src_fps:.1f}  |  Resolution: '
          f'{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
    print(f'Strategy: {strategy}  |  Predict every {predict_every} frames  |  '
          f'Smooth window: {smooth_window}')

    # ── state ─────────────────────────────────────────────────────────
    frame_buffer   = collections.deque(maxlen=buffer_size)    # raw BGR frames
    pred_history   = collections.deque(maxlen=smooth_window)  # recent label strings
    timestamps_log = []                                        # (frame_no, time_s, label, conf)

    label, confidence, all_probs = '...', 0.0, np.ones(NUM_CLASSES) / NUM_CLASSES
    smoothed_label = None

    frame_no     = 0
    t_prev       = time.perf_counter()
    fps_display  = 0.0

    if display_inline:
        img_widget = widgets.Image(format='jpeg')
        display(img_widget)

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print('End of video / stream.')
                break
            if max_frames and frame_no >= max_frames:
                print(f'Reached max_frames={max_frames}. Stopping.')
                break

            frame_buffer.append(frame)  # raw BGR pushed to sliding window

            # ── run prediction every N frames once buffer has enough data
            if frame_no % predict_every == 0 and len(frame_buffer) >= NUM_FRAMES:
                label, confidence, all_probs = predict_shot(frame_buffer)

                # majority-vote smoothing
                pred_history.append(label)
                smoothed_label = collections.Counter(pred_history).most_common(1)[0][0]

                # log timestamp if above threshold
                if save_timestamps and confidence >= conf_threshold:
                    t_sec = round(frame_no / src_fps, 3)
                    timestamps_log.append((frame_no, t_sec, label, round(confidence*100, 2)))

            # ── FPS calculation ───────────────────────────────────────
            t_now = time.perf_counter()
            fps_display = 0.9 * fps_display + 0.1 * (1.0 / max(t_now - t_prev, 1e-6))
            t_prev = t_now

            # ── overlay ───────────────────────────────────────────────
            vis_frame = draw_overlay(
                frame, label, confidence, fps_display, all_probs, smoothed_label)

            # ── display ───────────────────────────────────────────────
            if display_inline:
                if frame_no % inline_every == 0:
                    rgb = cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB)
                    pil_img = Image.fromarray(rgb)
                    buf = io.BytesIO()
                    pil_img.save(buf, format='JPEG', quality=75)
                    img_widget.value = buf.getvalue()
            else:
                cv2.imshow('Cricket Shot Detection — Press Q to quit', vis_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print('User stopped.')
                    break

            frame_no += 1

    except KeyboardInterrupt:
        print('Interrupted by user.')
    finally:
        cap.release()
        if not display_inline:
            cv2.destroyAllWindows()

    print(f'\nProcessed {frame_no} frames.')

    # ── save timestamp log ────────────────────────────────────────────
    if save_timestamps and timestamps_log:
        import csv
        with open(csv_path, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['frame', 'time_s', 'shot', 'confidence_%'])
            w.writerows(timestamps_log)
        print(f'Shot timestamps saved → {csv_path} ({len(timestamps_log)} events)')

    return timestamps_log


print('✓ run_realtime ready')

In [ ]:
# ── Cell 10: RUN ───────────────────────────────────────────────────────
# Kaggle: display_inline=True (no GUI)
# Local with monitor: display_inline=False (cv2.imshow)

timestamps = run_realtime(
    source         = VIDEO_SOURCE,
    predict_every  = PREDICT_EVERY_N_FRAMES,
    buffer_size    = BUFFER_SIZE,
    conf_threshold = CONFIDENCE_THRESHOLD,
    smooth_window  = SMOOTH_WINDOW,
    strategy       = SAMPLING_STRATEGY,
    display_inline = True,   # ← change to False for local cv2.imshow
    max_frames     = None,   # ← set e.g. 300 to cap at 300 frames for testing
)

In [ ]:
# ── Cell 11: Post-run analysis of detected shots ───────────────────────
import pandas as pd
import matplotlib.pyplot as plt

if timestamps:
    df = pd.DataFrame(timestamps, columns=['frame', 'time_s', 'shot', 'confidence_%'])
    print(f'Total shot events logged: {len(df)}')
    print('\nShots detected:')
    print(df['shot'].value_counts().to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Shot distribution
    df['shot'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Shot Frequency', fontsize=13)
    axes[0].set_xlabel('Shot Type')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)

    # Confidence over time
    axes[1].plot(df['time_s'], df['confidence_%'], marker='o', markersize=3,
                 color='darkorange', linewidth=1.2)
    axes[1].axhline(CONFIDENCE_THRESHOLD * 100, color='red', linestyle='--',
                    linewidth=1, label=f'Threshold {CONFIDENCE_THRESHOLD*100:.0f}%')
    axes[1].set_title('Confidence Over Time', fontsize=13)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Confidence (%)')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('/kaggle/working/shot_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved → /kaggle/working/shot_analysis.png')

    display(df.head(20))
else:
    print('No events logged (check CONFIDENCE_THRESHOLD or VIDEO_SOURCE).')

In [ ]:
# ── Cell 12: BONUS — Gradio real-time UI (upload video → live output) ──
# Uncomment to install:
# import subprocess; subprocess.check_call(['pip', 'install', '-q', 'gradio'])

import gradio as gr
import tempfile


def gradio_predict(video_file):
    """
    Called by Gradio when the user uploads a video.
    Returns: (annotated video path, shot timeline markdown)
    """
    cap = cv2.VideoCapture(video_file)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    fw  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_path  = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    fourcc    = cv2.VideoWriter_fourcc(*'mp4v')
    writer    = cv2.VideoWriter(out_path, fourcc, src_fps, (fw, fh))

    frame_buffer  = collections.deque(maxlen=BUFFER_SIZE)
    pred_history  = collections.deque(maxlen=SMOOTH_WINDOW)
    timeline_rows = []

    label, confidence, all_probs = '...', 0.0, np.ones(NUM_CLASSES) / NUM_CLASSES
    smoothed_label = None
    frame_no = 0

    with torch.inference_mode():
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame_buffer.append(frame)

            if frame_no % PREDICT_EVERY_N_FRAMES == 0 and len(frame_buffer) >= NUM_FRAMES:
                label, confidence, all_probs = predict_shot(frame_buffer)
                pred_history.append(label)
                smoothed_label = collections.Counter(pred_history).most_common(1)[0][0]
                if confidence >= CONFIDENCE_THRESHOLD:
                    timeline_rows.append(
                        f'| {frame_no/src_fps:.2f}s | **{smoothed_label}** | {confidence*100:.1f}% |')

            vis = draw_overlay(frame, label, confidence, 0.0, all_probs, smoothed_label)
            writer.write(vis)
            frame_no += 1

    cap.release()
    writer.release()

    timeline_md  = '| Time | Shot | Confidence |\n|------|------|------------|\n'
    timeline_md += '\n'.join(timeline_rows[:50]) if timeline_rows else '| — | No shots detected | — |'

    return out_path, timeline_md


demo = gr.Interface(
    fn          = gradio_predict,
    inputs      = gr.Video(label='Upload Cricket Video'),
    outputs     = [
        gr.Video(label='Annotated Output'),
        gr.Markdown(label='Shot Timeline'),
    ],
    title       = '🏏 Cricket Shot Detector — Real-Time',
    description = (
        f'EfficientNetB0 + {ARCH.upper()} | '
        f'{SAMPLING_STRATEGY} sampling | '
        f'Classes: {", ".join(CLASSES)}'
    ),
    theme       = gr.themes.Soft(),
)

demo.launch(share=True)  # share=True gives a public ngrok URL on Kaggle

---
## Quick Usage Guide

| Cell | What to change | Default |
|------|---------------|--------|
| 2 | `CHECKPOINT_PATH` | your `.pth` path |
| 2 | `VIDEO_SOURCE` | video file or `0` for webcam |
| 2 | `SAMPLING_STRATEGY` | `'uniform'` (fastest), `'motion'`, `'hybrid'` |
| 2 | `CONFIDENCE_THRESHOLD` | `0.40` |
| 2 | `PREDICT_EVERY_N_FRAMES` | `8` — lower = more responsive |
| 2 | `SMOOTH_WINDOW` | `5` — majority vote window |
| 10 | `display_inline` | `True` for Kaggle, `False` for local cv2 window |
| 10 | `max_frames` | `None` or integer to limit processing |

### Choosing a sampling strategy
- **uniform** — fastest; good for real-time
- **motion** — focuses on high-motion frames; slightly slower
- **hybrid** — mix of motion + uniform + random; best quality, ~20% slower

### Loading a specific architecture
Set `ARCH_OVERRIDE` in Cell 2 if your checkpoint filename does not contain `lstm`, `gru`, or `cnn_only`.